# Busa, the Concordance, and Where All This Came From

**Computational Text Analysis Pack · Module 21 — The Lineage**

*Technique:* Index verborum and KWIC as the founding method; reconstructing the workflow

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Sagaustus/adh-dh-notebooks/blob/main/00-foundations/21a_busa_and_the_concordance.ipynb)

Data: `colonial_texts.csv` — from the `data/` folder of this pack.

---

In [ ]:
# Setup — run this first.
import os, warnings
warnings.filterwarnings('ignore')
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from collections import Counter

# The data this notebook needs. It is downloaded from CourseHub automatically;
# uploading by hand is only a fallback if that server is unreachable.
NEEDED = ['colonial_texts.csv']
DATA_URL = 'https://coursehub.africandigitalhumanities.ca/ide/pack-data/text/'

def _missing():
    return [f for f in NEEDED if not os.path.exists(f)]

missing = _missing()
if missing:
    # Fetch from CourseHub first, so nobody has to upload anything.
    import urllib.request, urllib.parse
    for _f in list(missing):
        try:
            urllib.request.urlretrieve(DATA_URL + urllib.parse.quote(_f), _f)
            print(f'downloaded {_f}')
        except Exception as _e:
            print(f'could not fetch {_f}: {_e}')
    missing = _missing()

if missing:
    try:
        from google.colab import files
    except ImportError:
        raise SystemExit('Place these next to the notebook: ' + ', '.join(missing))
    # Ask again until everything has arrived. The upload widget returns as soon
    # as you close it, so picking only some of the files would otherwise fail a
    # few lines below with a confusing FileNotFoundError.
    for _ in range(4):
        print('Select ALL of these at once (ctrl-click / cmd-click to multi-select):')
        print('   ' + ', '.join(missing))
        files.upload()
        missing = _missing()
        if not missing:
            break
        print('Still needed: ' + ', '.join(missing))
    if missing:
        raise SystemExit(
            'Missing: ' + ', '.join(missing) + '. Re-run this cell and select '
            'every file listed, or upload them with the folder icon on the left.')

df_colonial = pd.read_csv('colonial_texts.csv')
NIGERIAN = ['Hausa', 'Igbo', 'Yoruba', 'Nigerian Pidgin']
# Most of this pack runs on scikit-learn and the standard library. Also
# available: nltk, spacy, gensim and networkx (modules 14-19), Keras
# (case studies), and a GPU pod reached through /ide/api/neural/ for the
# neural modules — never called directly, so no token lives in a notebook.

plt.rcParams['figure.figsize'] = (10, 5.5)
plt.rcParams['axes.grid'] = True
plt.rcParams['grid.alpha'] = 0.25

print(f'Loaded. df_colonial has {len(df_colonial):,} rows and {df_colonial.shape[1]} columns.')

## What this notebook does

In 1946 a Jesuit priest, Roberto Busa, met Thomas J. Watson Sr. of IBM and
secured backing for an index of every word in the complete works of Thomas
Aquinas. Work began around 1949 on punched-card tabulating equipment. The
*Index Thomisticus* was finished in **56 printed volumes by 1980** — 34 years,
**10,631,980 words**, covering Aquinas's 118 texts plus 61 related texts by
other authors, every word morphologically tagged and lemmatised.

That project is why ADHO's lifetime achievement award is the Busa Prize, and it
is the direct ancestor of every method in this course. This notebook rebuilds
its core operation, at a scale Busa would have found trivial and with an effort
he would have found insulting.

## A necessary correction to the story

The usual telling is Busa-and-Watson, two great men. Recent historiography has
recovered the **women who did the actual work** — the keypunch operators, trained
in Busa's school at Gallarate, who transferred millions of words onto cards and
whose names went unrecorded for decades.

This is not a footnote to the history of DH method; it is the history. The
labour that produces a corpus is usually invisible, usually gendered, and
usually uncredited, and that pattern did not end in 1980. Susan Hockey's chapter
in the *Companion to Digital Humanities* (2004) is the standard account of the
period; the recovery work complicates it in the right direction.

In [ ]:
import re

def tokenise(text):
    """Words, lowercased, Unicode-aware.

    `\\w+` matches Unicode letters, so ọ, à, ƙ, ɗ and ɓ survive. The common
    shortcut `[a-z]+` would throw them away and quietly change every count in
    this notebook — see notebook 03b.
    """
    return re.findall(r"\w+", str(text).lower())

sample = "Àwọn ọmọ ìlú ti ṣe ìpàdé — ƙungiyar ɗalibai ta yi taro"
print("sample     :", sample)
print("tokens     :", tokenise(sample))
print("naive [a-z]:", re.findall(r"[a-z]+", sample.lower()))
print()
print("The naive pattern splits words at every accented letter.")

In [ ]:
# The index verborum: every word, every location. This is the whole idea.
work = (df_colonial.groupby(["work_id", "title", "author"])["text"]
        .apply(" ".join).reset_index())
chosen = work.sort_values("text", key=lambda s: s.str.len(), ascending=False).iloc[0]
tokens = tokenise(chosen["text"])
print(f"'{chosen['title'][:56]}' — {len(tokens):,} words")

index = {}
for position, word in enumerate(tokens):
    index.setdefault(word, []).append(position)
print(f"index verborum: {len(index):,} distinct words, "
      f"{sum(len(v) for v in index.values()):,} locations")
print()
print(f"For scale: Busa's team did this for {10_631_980:,} words of medieval")
print(f"Latin on punched cards. This cell did "
      f"{len(tokens):,} in a fraction of a second.")

## KWIC: the instrument the index makes possible

Once you have every location, you can print each occurrence with its context
lined up in a column — **keyword in context**. Module 5 taught KWIC as a
technique. Here it is as the founding instrument: the thing an index is *for*.

In [ ]:
def kwic(word, width=38, limit=10):
    out = []
    for pos in index.get(word, [])[:limit]:
        left = " ".join(tokens[max(0, pos - 12):pos])[-width:]
        right = " ".join(tokens[pos + 1:pos + 13])[:width]
        out.append(f"{left:>{width}}  {tokens[pos].upper():^12}  {right:<{width}}")
    return out

TERM = "native"        # <- change this
print(f"KWIC concordance for '{TERM}' "
      f"({len(index.get(TERM, []))} occurrences)\n")
for line in kwic(TERM):
    print(line)

## Lemmatisation, and why Busa needed it

An index of surface forms scatters a word across its inflections. Busa's team
lemmatised everything — by hand, with a morphological analysis of Latin — because
an index that separates *amare*, *amat* and *amavit* does not answer the
question a philosopher is asking.

Module 14 built the modern versions of that operation. This is what they are for.

In [ ]:
import nltk
from nltk.stem import WordNetLemmatizer

lem = WordNetLemmatizer()
lemma_index = {}
for word, positions in index.items():
    key = lem.lemmatize(lem.lemmatize(word, "v"), "n")
    lemma_index.setdefault(key, []).extend(positions)

print(f"surface forms : {len(index):,}")
print(f"lemmas        : {len(lemma_index):,}")
print(f"collapse      : {1 - len(lemma_index)/len(index):.1%}\n")

biggest = sorted(((len(set(w for w in index if
                           lem.lemmatize(lem.lemmatize(w, 'v'), 'n') == k)), k)
                  for k in list(lemma_index)[:3000]), reverse=True)[:8]
for n, k in biggest:
    forms = sorted(w for w in index
                   if lem.lemmatize(lem.lemmatize(w, "v"), "n") == k)
    print(f"   {k:<14}{n} forms: {', '.join(forms[:6])}")

In [ ]:
# Dispersion: where in the work a word falls. Busa's cards could be sorted;
# this is the same operation, and module 10 gave it a statistic.
fig, ax = plt.subplots(figsize=(9, 2.6))
for i, term in enumerate([TERM, "god", "king"]):
    pos = index.get(term, [])
    if pos:
        ax.scatter(pos, [i] * len(pos), s=6, alpha=.55, label=f"{term} ({len(pos)})")
ax.set_yticks(range(3))
ax.set_yticklabels([TERM, "god", "king"])
ax.set_xlabel("word position in the text")
ax.set_title("Dispersion — the sorted-card view")
ax.legend(fontsize=8, loc="upper right")
plt.tight_layout()

print("\nThe through-line the survey identifies — concordance, KWIC, frequency")
print("list, lemma — is every method in modules 2, 5, 10 and 14 of this course.")
print("The instruments changed. The questions did not.")

### 🔧 Your turn 1

Pick a term central to your own research and read fifty KWIC lines before running any statistic on it. What did you learn that a frequency count would not have told you?

<details markdown="1">
<summary><b>What you should have seen</b> — click to expand</summary>

Almost always: that the word is doing several different jobs, and that a count
had silently averaged over them. This is the reason the concordance survived
seventy years of methodological change and why module 11 insists a count is a
pointer to reading rather than a substitute for it. Busa's project was not
built to replace reading Aquinas — it was built so that reading Aquinas could
be exhaustive rather than selective, which is a different and more modest claim
than most distant-reading rhetoric makes.

</details>